# Урок 13 - Памет на агента с графи на знания Cognee


## Setup

This notebook demonstrates how to build an intelligent **coding assistant** with persistent memory using [**Cognee**](https://www.cognee.ai/) knowledge graphs and the **Microsoft Agent Framework** (MAF).

Cognee transforms unstructured text into a structured, queryable knowledge graph backed by vector embeddings — giving your agent a rich, relationship-aware long-term memory.

### What You'll Learn
1. **Build Knowledge Graphs**: Transform developer profiles and best practices into structured, queryable knowledge.
2. **Integrate Cognee with MAF**: Use `@tool` functions to let an MAF agent query Cognee's knowledge graph.
3. **Session-Aware Conversations**: Maintain context across multiple questions in the same session.
4. **Long-Term Memory**: Persist important knowledge across sessions and retrieve it in new conversations.

### Prerequisites
- Python 3.9+
- Redis running locally (`docker run -d -p 6379:6379 redis`) for session management
- An LLM API key (e.g. OpenAI) — set `LLM_API_KEY` in `.env`
- `CACHING=true` in `.env` (required for Cognee sessions)
- A Microsoft Foundry project with a deployed chat model
- `AZURE_AI_PROJECT_ENDPOINT` and `AZURE_AI_MODEL_DEPLOYMENT_NAME` in `.env`
- Azure CLI authenticated (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")


In [ ]:
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)

print("✅ FoundryChatClient created")


## Видове памет на агента

Този тетрадка изследва същите трите типа памет от основната тетрадка на Урок 13, но използва Cognee като бекенд за дългосрочна памет:

| Тип памет | Механизъм | Продължителност |
|---|---|---|
| **Работна** | `agent.create_session()` (MAF) | Еднократен разговор |
| **Краткосрочна** | Кеш за сесия Cognee (Redis) | Една сесия |
| **Дългосрочна** | Знаниева графа и вектори Cognee | Постоянна |

### Архитектура на паметта на Cognee
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## Подгответе съхранението на Cognee


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## Част 1 — Създаване на Базата с Познания

Вземаме три вида данни, за да създадем всеобхватна база с познания за нашия помощник при кодирането:

1. **Профил на разработчика** — лична експертиза и технически опит
2. **Най-добри практики в Python** — Зенът на Python с практически насоки
3. **Исторически разговори** — минали сесии с въпроси и отговори между разработчици и AI асистенти


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## Визуализирайте графа на знанието

Cognee може да изобрази интерактивна HTML визуализация на извлечените обекти и връзки.


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## Обогатете паметта с Memify

`memify()` анализира графа на знанията и генерира интелигентни правила — идентифицирайки модели, добри практики и взаимоотношения между концепции.


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## Част 2 — MAF агент с Cognee инструменти

Сега създаваме MAF агент, който може да запитва графа на знанието на Cognee чрез функции `@tool`. Това позволява на агента да използва пълната мощ на семантичното търсене, осъзнаващо графа, като същевременно поддържа контекста на разговора чрез сесии.


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = provider.as_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")


## Работна памет с сесии

`AgentSession` (създаден чрез `agent.create_session()`) осигурява работна памет в рамките на сесията. Агентът може да се позовава на по-ранни съобщения, като същевременно запитва дългосрочния граф на знанието на Cognee.


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## Нова сесия — Дългосрочната памет се запазва

Започването на нова сесия изчиства работната памет, но графът на знанията Cognee все още е достъпен. Актьорът може да извлече същите дългосрочни знания в изцяло нов разговор.


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## Резюме

В тази тетрадка изградихте помощник за кодиране, който комбинира **работната памет на MAF** (`agent.create_session()`) с **дългосрочния граф на знанието на Cognee**.

### Какво научихте
1. **Създаване на граф на знанието**: Cognee приема неструктуриран текст и изгражда граф + векторна памет.
2. **Обогатяване на графа с memify**: Изведени факти и по-богати връзки върху съществуващия ви граф.
3. **Интеграция на MAF + Cognee**: Функциите `@tool` позволяват на агентите MAF да запитват графа на Cognee естествено.
4. **Работна памет + дългосрочна памет**: `AgentSession` (чрез `agent.create_session()`) осигурява контекст на сесията, докато Cognee предоставя устойчива памет.
5. **Филтрирано търсене с NodeSets**: Насочване към конкретни подмножества от графа на знанието (например само принципи).

### Основни изводи
- **Cognee** превръща суровия текст в структурирана, връзко-осъзнатa памет — по-мощна от плоско векторно хранилище.
- Функциите **`@tool`** чисто свързват агентите MAF и външни системи за знания.
- **`AgentSession`** (чрез `agent.create_session()`) държи контекста на разговор отделен от дългосрочните знания.
- Един и същ граф на знанието обслужва множество сесии и агенти.

### Приложения в реалния свят
- **Копилоти за разработчици**: Преглед на код, анализ на инциденти, помощници за архитектура
- **Копилоти за клиенти**: Поддържащи агенти върху продуктова документация, често задавани въпроси и бележки в CRM
- **Вътрешни експертни копилоти**: Помощници за политики, юридически или сигурност, разсъждаващи върху насоки
- **Обединени слоеве данни**: Комбиниране на структурирани и неструктурирани данни в един граф, който може да се запитва

### Следващи стъпки
- Експериментиране с времева осъзнатост в Cognee
- Дефиниране на OWL онтология за качеството на домейн-специфичния граф
- Добавяне на цикли за обратна връзка от потребители за подобряване на извличането с времето
- Масшабиране до мулти-агентни системи, споделящи един и същ слой памет на Cognee


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Отказ от отговорност**:
Този документ е преведен с помощта на AI преводачески услуга [Co-op Translator](https://github.com/Azure/co-op-translator). Въпреки че се стремим към точност, моля имайте предвид, че автоматизираните преводи могат да съдържат грешки или неточности. Оригиналният документ на неговия роден език трябва да се счита за авторитетен източник. За критична информация се препоръчва професионален човешки превод. Ние не носим отговорност за каквито и да е недоразумения или неправилни тълкувания, произтичащи от използването на този превод.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
